# Agent Planning

This notebook adds a safe, additive tutorial on planning. It does not change the main workflow. Instead, it teaches why planning matters, how ReAct differs from planner-executor designs, and how task decomposition can make agent behavior easier to inspect.

## Learning goals

- Understand why agent planning matters.
- Compare ReAct and planner-executor patterns.
- See how task decomposition changes a plan.
- Execute a simple plan with explicit handlers.


## Concept explanation

As in the other notebooks, we first verify the active interpreter. This helps confirm that the notebook is running inside the uv-managed environment registered by the setup script.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

This setup cell imports the additive planning helpers from `src/planner_extended.py` without touching the existing `src/planner.py` module. The notebook stays tutorial-focused while the reusable logic remains in `src/`.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.planner_extended import PlanExecutor, PlanGenerator

pd.set_option('display.max_colwidth', 140)
generator = PlanGenerator()


Planning helps an agent break a vague request into smaller, more manageable actions. Without a plan, an agent often jumps directly to synthesis and makes it harder to debug where mistakes came from.

ReAct emphasizes an iterative loop of observe, think, act, and observe again. A planner-executor pattern separates planning from action execution. Task decomposition focuses on splitting a broad request into smaller sub-tasks that can be solved more reliably.

## Implementation


In [ ]:
planning_task = 'Find the rollout date, compare it to the pilot window, and summarize why the timing matters.'
strategy_comparison = generator.compare_strategies(planning_task)
{
    strategy: len(plan)
    for strategy, plan in strategy_comparison.items()
}


The table below makes the different planning styles concrete. ReAct tends to produce reasoning-oriented steps, planner-executor produces a clean execution outline, and decomposition breaks the task into direct sub-problems.


In [ ]:
plan_frames = {
    strategy: pd.DataFrame(plan)
    for strategy, plan in strategy_comparison.items()
}
plan_frames['react'], plan_frames['planner_executor'], plan_frames['decompose']


A planner becomes more useful when it can be paired with an executor. The executor below uses explicit handlers so the notebook can show exactly what happens when a step requests retrieval or a tool.


In [ ]:
executor = PlanExecutor()
executor.register_handler('retrieval', lambda step: {'status': 'completed', 'output': f"Retrieved evidence for: {step['objective']}"})
executor.register_handler('tool', lambda step: {'status': 'completed', 'output': f"Ran a focused tool for: {step['objective']}"})
planner_executor_plan = generator.generate_plan(planning_task, strategy='planner_executor')
execution_log = executor.execute(planner_executor_plan)
pd.DataFrame(execution_log)


## Experiment

A useful planning experiment is to compare how the plan changes when the task becomes simpler or more multi-step. This cell generates multiple plans so you can see how decomposition reacts to task structure.


In [ ]:
experiment_tasks = [
    'Summarize the rollout plan.',
    'Find the rollout date and explain who needs to know it.',
    'Search the policy goals, calculate the pilot duration, and draft a short update.',
]
experiment_rows = []
for task in experiment_tasks:
    plan = generator.generate_plan(task, strategy='decompose')
    experiment_rows.append({'task': task, 'plan_length': len(plan), 'first_step': plan[0]['objective']})
pd.DataFrame(experiment_rows)


## Result analysis

The planning results show that a good agent does not need one universal planning style. ReAct is useful for iterative reasoning, planner-executor is useful when you want clean boundaries between planning and action, and decomposition is useful when a task clearly contains several sub-parts.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'strategy': 'react', 'strength': 'good for iterative observe-think-act loops'},
        {'strategy': 'planner_executor', 'strength': 'good for explicit planning and safe execution'},
        {'strategy': 'decompose', 'strength': 'good for turning broad tasks into manageable chunks'},
    ]
)
analysis_frame


## Takeaways

- Planning makes agent behavior easier to inspect and debug.
- ReAct, planner-executor, and task decomposition serve different purposes.
- Explicit execution logs help you separate planning quality from tool quality.
- This notebook adds planning education without changing the existing repo architecture.
